In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np
from pathlib import Path
import glob
import h5py

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology
from skewerjax.simulations.simulations import TNGSimulation

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux

import skewerjax.optical_depth.optical_depth as od

from skewerjax.constants.physical_constants import phys

In [ ]:
from fake_spectra.spectra import Spectra

In [ ]:
wavelengths = jnp.linspace(970, 1230, 2**16)  # in Angstroms

In [ ]:
base = "/home/STORAGE/TNG50-4"
num  = 25

sim = TNGSimulation(base)

In [ ]:
H0 = 100* sim.cosmo["h"] * 1e6 * phys.Gyr_to_s / phys.Gpc_to_m # [1/Gyr]

Omega_m_0 = sim.cosmo["Omega_m"] + sim.cosmo["Omega_b"]
Omega_k_0 = 0
Omega_L_0 = sim.cosmo["Omega_L"]
Omega_r_0 = 1 - (Omega_m_0 + Omega_k_0 + Omega_L_0)

L_box_ckpc_over_h = sim.cosmo["BoxSize"] # comoving kpc/h
L_box_Mpc = L_box_ckpc_over_h * 1e-3 / sim.cosmo["h"]  # comoving Mpc

a0 = 1 / (1 + sim.snapshots[num].header["Redshift"])

In [ ]:
# Create cosmology object
cosmology = Cosmology(
    H0=H0,
    Omega_r_0=Omega_r_0,
    Omega_m_0=Omega_m_0,
    Omega_k_0=Omega_k_0,
    Omega_lambda_0=Omega_L_0,
)

# Define which cosmology to use, this is set as default to 0 since skewer and cosmology objects can deal with multiple cosmologies at once, 
# but other classes such as fields and optical depth are currently only set up for a single cosmology at a time

ii_cosmology = 0

In [ ]:
N_grid = 2
N_ppsk = 16

N_sk = N_grid**2

_points = aux.make_skewer_grid_points(L_box=L_box_Mpc, N_grid=N_grid, N_ppsk=N_ppsk, direction="y")
_points = jnp.reshape(_points, (N_sk, N_ppsk, 3))

skewer_bundle = SkewerBundle(
    spatial_points=_points,
    a0=a0,
    x0=0.0,
    ICs_defined_at_index=-1,
    ICs_at_emission_or_absorption="absorption",
    cosmologies=cosmology
)

In [ ]:
P = np.asarray(skewer_bundle.spatial_points)   # (N_sk, N_ppsk, 3)
D = np.asarray(skewer_bundle.los_directions)   # (N_sk, N_ppsk, 3)

# --- axis: 1-indexed (1=x, 2=y, 3=z) ---
d0 = D[:, 0, :]                               # direction per skewer (take first point)
axis0 = np.argmax(np.abs(d0), axis=1)         # 0=x,1=y,2=z
axis = (axis0 + 1).astype(np.int32)

# Sanity: directions constant along each skewer and axis-aligned (one-hot-ish)
tol = 1e-6
if np.max(np.abs(D - D[:, :1, :])) > tol:
    raise ValueError("los_directions is not constant along the points of (at least) one skewer.")
if not np.all(np.sum(np.abs(d0) > 0.5, axis=1) == 1):
    raise ValueError("Some skewers are not axis-aligned; expected one-hot los_directions like [0,1,0].")

# --- cofm: transverse position per skewer (column along LOS axis is ignored by fake_spectra) ---
cofm = P[:, 0, :].astype(np.float64)          # take first point of each skewer as the transverse anchor
for i in range(N_sk):
    cofm[i, axis0[i]] = 0.0                   # optional: set ignored coordinate to 0 for cleanliness

In [ ]:
tmp_savedir = Path("/tmp/fake_spectra_inmemory")
tmp_savedir.mkdir(parents=True, exist_ok=True)
sp = Spectra(
    num, base, cofm=cofm, axis=axis, kernel="tophat", nbins=1024, res=None,
    MPI=None, savedir=str(tmp_savedir), savefile="dummy_not_written.hdf5", reload_file=True,
)

In [ ]:
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206", "SiII-1190", "SiII-1193"])

In [ ]:
tau = {}
n = {}       # number density [atoms/cm^3] or special channels
T = {}       # temperature [K]
vlos = {}    # LOS velocity [km/s]

roman_to_ion = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5, "VI": 6, "VII": 7, "VIII": 8}

for species in line_bundle.species_keys:
    tau[species] = {}

    # Parse (elem, ion) from species string, e.g. HI -> (H,1), SiIII -> (Si,3)
    s = species
    roman = None
    for cand in sorted(roman_to_ion.keys(), key=len, reverse=True):
        if s.endswith(cand):
            roman = cand
            break
    if roman is None:
        raise ValueError(f"Cannot parse ion stage from species='{species}'")

    elem = s[: -len(roman)]
    if elem == "":
        raise ValueError(f"Cannot parse element symbol from species='{species}'")
    ion = roman_to_ion[roman]

    for line in line_bundle.lines_for_species(species):
        lam_key = float(line.lambda0_A)          # keep physical lambda0 as dict key
        lam_call = int(np.floor(lam_key))        # fake_spectra wants floor(lambda)
        tau[species][lam_call] = sp.get_tau(elem, ion, lam_call, force_recompute=True)

    n[species] = sp.get_density(elem, ion)
    T[species] = sp.get_temp(elem, ion)
    vvec = sp.get_velocity(elem, ion)  # (N, nbins, 3)
    vlos[species] = np.vstack([vvec[i, :, axis[i]-1] for i in range(N_sk)])  # (N, nbins)

In [ ]:
print("tau keys:", {spk: sorted(inner.keys()) for spk, inner in tau.items()})

In [ ]:
# Coordinate along the skewer in velocity units (km/s)
v_kms = (np.arange(sp.nbins) + 0.5) * sp.dvbin

# Optional physical distance (Mpc, physical)
MPC_IN_CM = 3.085678e24
x_Mpc_phys = (v_kms / sp.velfac) * (sp.rscale / MPC_IN_CM)

In [ ]:
ii_skewer = 0

species_to_plot = [s for s in line_bundle.species_keys if s in n and s in T and s in vlos]

cmap = plt.get_cmap("tab10")
colors = {spk: cmap(i % cmap.N) for i, spk in enumerate(species_to_plot)}

linestyles = ["-", "--", ":"]

fig, axes = plt.subplots(5, 1, figsize=(11, 13), sharex=True)

line_handles = []
for species in species_to_plot:
    axes[0].plot(x_Mpc_phys, n[species][ii_skewer], color=colors[species], linewidth=1.3)
    axes[1].plot(x_Mpc_phys, T[species][ii_skewer], color=colors[species], linewidth=1.3)
    axes[2].plot(x_Mpc_phys, vlos[species][ii_skewer], color=colors[species], linewidth=1.3)

    for j, lam in enumerate(tau[species].keys()):
        line_handles.append(Line2D([0], [0], color="grey", lw=2, linestyle=linestyles[j], label=f"{species} {lam}"))
        axes[3].plot(x_Mpc_phys, tau[species][lam][ii_skewer], color=colors[species], linestyle=linestyles[j], linewidth=1.3)
        axes[4].plot(x_Mpc_phys, np.exp(-tau[species][lam][ii_skewer]), color=colors[species], linestyle=linestyles[j], linewidth=1.3)

axes[0].set_ylabel(r"$n$ [cm$^{-3}$]")
axes[0].set_yscale("log")

axes[1].set_ylabel(r"$T$ [K]")
axes[1].set_yscale("log")

axes[2].set_ylabel(r"$v_{\parallel}$ [km/s]")


axes[3].set_ylabel(r"$\tau$")
axes[3].set_yscale("log")

axes[4].set_ylabel(r"$F/F_0$")
axes[4].set_xlabel(r"Distance along LOS [Mpc (physical)]")

species_handles = [
    Line2D([0], [0], color=colors[spk], lw=6, linestyle="-", label=spk)
    for spk in species_to_plot
]
leg1 = axes[0].legend(handles=species_handles, loc="upper right", frameon=True, fontsize=14, title="Species")
axes[0].add_artist(leg1)

leg2 = axes[-1].legend(handles=line_handles, loc="upper left", frameon=True, fontsize=12, title="Lines")
axes[-1].add_artist(leg2)

plt.tight_layout()
plt.show()

In [ ]:
fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_tiny, T0_tiny, L_box_Kpc)),     # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box_Kpc)),                   # UNITS: [km/s]
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min, n0_max, L_box_Kpc)), # UNITS: [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                        # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                       # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_tiny, T0_tiny, L_box_Kpc)),     # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box_Kpc)),                   # UNITS: [km/s]
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min, n0_max, L_box_Kpc)), # UNITS: [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                        # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))                       # UNITS: [GHz]
    }
})

In [ ]:
optical_depth = od.compute_optical_depth(skewer_bundle, wavelengths, line_bundle, fields)
F_over_F0 = jnp.exp(-optical_depth)